# Notebook 4: การสร้างชุดข้อมูลขั้นสุดท้าย
# (Building Final Datasets)

## สิ่งที่จะได้เรียนรู้
- สร้าง RAG dataset พร้อม embeddings
- ตรวจสอบคุณภาพ SFT dataset
- วิเคราะห์สถิติชุดข้อมูล
- ขั้นตอนถัดไป: การนำไปใช้จริง

In [ ]:
# ติดตั้ง dependencies (Install dependencies)
%pip install sentence-transformers pandas matplotlib numpy

# แก้ปัญหา symlink บน Windows
import os, sys
if sys.platform == "win32":
    os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
    import huggingface_hub.file_download as _hf_dl, shutil
    _orig = _hf_dl._create_symlink
    def _copy_fallback(src, dst, new_blob=False):
        try:
            _orig(src, dst, new_blob=new_blob)
        except OSError:
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            if os.path.exists(dst):
                os.remove(dst)
            shutil.copy2(src, dst)
    _hf_dl._create_symlink = _copy_fallback
    print("Applied Windows symlink workaround")

In [ ]:
import json
from pathlib import Path

# โหลด SFT dataset
sft_path = Path("../output/datasets/sft_dataset.jsonl")
sft_data = []
with open(sft_path, "r", encoding="utf-8") as f:
    for line in f:
        sft_data.append(json.loads(line))

# โหลด chunks
chunks_path = Path("../output/chunks/chunks.jsonl")
chunks = []
with open(chunks_path, "r", encoding="utf-8") as f:
    for line in f:
        chunks.append(json.loads(line))

print(f"SFT entries: {len(sft_data)}")
print(f"Chunks สำหรับ RAG: {len(chunks)}")

## สร้าง Embeddings สำหรับ RAG

**Embedding** คือการแปลงข้อความเป็นเวกเตอร์ตัวเลข เพื่อให้สามารถค้นหาข้อความที่คล้ายกันได้

เราจะใช้โมเดล `intfloat/multilingual-e5-small` ซึ่ง:
- รองรับหลายภาษารวมถึงภาษาไทย
- ขนาดเล็ก ทำงานได้บน CPU
- ให้ผลลัพธ์ดีสำหรับงาน retrieval

In [ ]:
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import numpy as np

# โหลดโมเดล embedding
print("กำลังโหลดโมเดล embedding...")
embed_model = SentenceTransformer("intfloat/multilingual-e5-small")
print("โหลดสำเร็จ!")

# สร้าง embeddings
texts = [chunk["text"] for chunk in chunks]

print(f"กำลังสร้าง embeddings สำหรับ {len(texts)} chunks...")
embeddings = embed_model.encode(
    texts,
    show_progress_bar=True,
    batch_size=32,
    normalize_embeddings=True
)

print(f"ขนาด embedding: {embeddings.shape}")
print(f"มิติ: {embeddings.shape[1]}")

In [ ]:
# สร้าง RAG dataset
rag_dataset = []

for i, (chunk, embedding) in enumerate(zip(chunks, embeddings)):
    rag_entry = {
        "chunk_id": chunk["chunk_id"],
        "text": chunk["text"],
        "metadata": chunk["metadata"],
        "embedding": embedding.tolist(),
    }
    rag_dataset.append(rag_entry)

# บันทึก
rag_path = Path("../output/datasets/rag_dataset.jsonl")
with open(rag_path, "w", encoding="utf-8") as f:
    for entry in rag_dataset:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"บันทึก RAG dataset: {rag_path}")
print(f"จำนวน: {len(rag_dataset)} entries")

In [ ]:
# ทดสอบค้นหาด้วย embedding (simple similarity search)
def search(query: str, top_k: int = 3) -> list[dict]:
    """ค้นหา chunks ที่เกี่ยวข้องกับ query"""
    query_embedding = embed_model.encode([query], normalize_embeddings=True)

    # คำนวณ cosine similarity
    similarities = np.dot(embeddings, query_embedding.T).flatten()

    # หา top-k
    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            "chunk_id": chunks[idx]["chunk_id"],
            "score": float(similarities[idx]),
            "text": chunks[idx]["text"][:200] + "...",
        })
    return results

# ทดสอบค้นหา
query = "ปัญญาประดิษฐ์"
print(f"ค้นหา: '{query}'\n")

results = search(query)
for r in results:
    print(f"[{r['score']:.4f}] {r['chunk_id']}")
    print(f"  {r['text']}")
    print()

In [ ]:
import pandas as pd

# ตรวจสอบ SFT dataset
df = pd.DataFrame(sft_data)

print("SFT Dataset Summary:")
print(f"  จำนวน entries: {len(df)}")
print(f"  Columns: {list(df.columns)}")
print()

# ตรวจค่าว่าง
print("ค่าว่าง:")
for col in df.columns:
    empty = df[col].apply(lambda x: len(str(x)) == 0).sum()
    print(f"  {col}: {empty} ว่าง")

# ความยาว
print("\nความยาว instruction:")
print(f"  เฉลี่ย: {df['instruction'].str.len().mean():.0f} ตัวอักษร")
print(f"  min: {df['instruction'].str.len().min()}")
print(f"  max: {df['instruction'].str.len().max()}")

print("\nความยาว output:")
print(f"  เฉลี่ย: {df['output'].str.len().mean():.0f} ตัวอักษร")
print(f"  min: {df['output'].str.len().min()}")
print(f"  max: {df['output'].str.len().max()}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

# ตั้งค่าให้แสดงภาษาไทย
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Instruction length distribution
axes[0].hist(df['instruction'].str.len(), bins=20, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_xlabel("Length (chars)")
axes[0].set_ylabel("Count")
axes[0].set_title("Instruction Length Distribution")

# 2. Output length distribution
axes[1].hist(df['output'].str.len(), bins=20, edgecolor='black', alpha=0.7, color='coral')
axes[1].set_xlabel("Length (chars)")
axes[1].set_ylabel("Count")
axes[1].set_title("Output Length Distribution")

# 3. RAG chunk length distribution
chunk_lengths = [len(c["text"]) for c in chunks]
axes[2].hist(chunk_lengths, bins=20, edgecolor='black', alpha=0.7, color='mediumseagreen')
axes[2].set_xlabel("Length (chars)")
axes[2].set_ylabel("Count")
axes[2].set_title("RAG Chunk Length Distribution")

plt.tight_layout()
plt.savefig("../output/datasets/dataset_statistics.png", dpi=150)
plt.show()

print("Dataset statistics saved!")

## สรุป

### ชุดข้อมูลที่สร้างได้:

| Dataset | Path | Format | จำนวน |
|---|---|---|---|
| SFT (Fine-tuning) | `output/datasets/sft_dataset.jsonl` | Alpaca JSONL | ดูข้างบน |
| RAG (Retrieval) | `output/datasets/rag_dataset.jsonl` | JSONL + embeddings | ดูข้างบน |

### ขั้นตอนถัดไป:

**สำหรับ Fine-tuning:**
- ใช้กับ [Unsloth](https://github.com/unslothai/unsloth) สำหรับ fine-tune Llama/Qwen
- อัปโหลดไป [Hugging Face Hub](https://huggingface.co/docs/datasets/)
- ใช้กับ [TRL](https://github.com/huggingface/trl) สำหรับ SFT training

**สำหรับ RAG:**
- ใช้กับ vector database เช่น ChromaDB, Milvus, Qdrant
- สร้าง retrieval pipeline ด้วย LangChain หรือ LlamaIndex

In [ ]:
# แสดงไฟล์ทั้งหมดที่สร้าง
output_base = Path("../output")

print("ไฟล์ทั้งหมดที่สร้างจาก Workshop:\n")
for subdir in ["extracted", "chunks", "datasets"]:
    dir_path = output_base / subdir
    if dir_path.exists():
        print(f"  {subdir}/")
        for f in sorted(dir_path.iterdir()):
            size_kb = f.stat().st_size / 1024
            print(f"    {f.name} ({size_kb:.1f} KB)")
        print()

print("Workshop เสร็จสิ้น!")